# 🗄️ Tiered Agent Memory Cache Lab
### L1 in-memory · L2 Redis · Pydantic validation · Eviction & monitoring

**The problem:** Every LLM call re-processes context it has already seen.
A tiered cache intercepts repeated or near-identical lookups before they hit the model.

```
Request
  └─> L1 (in-process dict, ~1ms)   HIT? return immediately
        └─> L2 (Redis, ~5ms)       HIT? promote to L1, return
              └─> LLM (Ollama)     compute, write to L2+L1, return
```

| Component | Role |
|-----------|------|
| L1 `OrderedDict` | Sub-millisecond, LRU, capped at N entries |
| L2 Redis | Persistent across restarts, TTL-based expiry |
| Pydantic `CacheEntry` | Validates every value written/read |
| `MemoryMonitor` | Tracks utilisation, triggers eviction at threshold |

> **Requirements:** Ollama running locally (`ollama serve`) with at least one model pulled.
> Redis is optional — the notebook detects availability and falls back gracefully.

## Section 0 — Setup & Connectivity Check

In [9]:
!pip install ollama redis pydantic sentence-transformers -q

import os, sys, time, json, hashlib, re
from collections import OrderedDict
from dataclasses import dataclass, field
from datetime import datetime, timezone, timedelta
from typing import Optional, Any
from enum import Enum
import threading
import numpy as np
from pydantic import BaseModel, Field, field_validator, ValidationError

# ── Ollama connectivity ───────────────────────────────────────────────────
import ollama as _ollama

OLLAMA_MODEL = 'llama3.2'   # change to any model you have pulled

def check_ollama() -> bool:
    try:
        models = _ollama.list()
        names  = [m.model for m in models.models]
        print(f'Ollama running. Available models: {names}')
        if not any(OLLAMA_MODEL in n for n in names):
            print(f'WARNING: {OLLAMA_MODEL!r} not found. Run: ollama pull {OLLAMA_MODEL}')
            return False
        return True
    except Exception as e:
        print(f'Ollama not reachable: {e}  ->  Start with: ollama serve')
        return False

def call_ollama(prompt: str, system: str = '', model: str = OLLAMA_MODEL) -> tuple[str, float]:
    msgs = []
    if system:
        msgs.append({'role': 'system', 'content': system})
    msgs.append({'role': 'user', 'content': prompt})
    t0   = time.time()
    resp = _ollama.chat(model=model, messages=msgs)
    return resp.message.content.strip(), time.time() - t0

# ── Redis connectivity ────────────────────────────────────────────────────
import redis as _redis

REDIS_AVAILABLE = False
redis_client    = None

def check_redis() -> bool:
    global REDIS_AVAILABLE, redis_client
    try:
        redis_client = _redis.Redis(host='localhost', port=6379, db=0,
                                     decode_responses=True, socket_connect_timeout=2)
        redis_client.ping()
        used = redis_client.info('memory').get('used_memory_human', 'N/A')
        print(f'Redis running. Memory used: {used}')
        REDIS_AVAILABLE = True
        return True
    except Exception as e:
        print(f'Redis not available ({e}). L2 will use in-process fallback.')
        REDIS_AVAILABLE = False
        redis_client    = None
        return False

ollama_ok = check_ollama()
redis_ok  = check_redis()
print(f'\nOllama: {"✅" if ollama_ok else "❌"}   Redis: {"✅" if redis_ok else "⚠️  fallback mode"}')


zsh:1: command not found: pip
Ollama running. Available models: ['mistral:latest', 'llama3.2:latest', 'llama3:latest', 'llava:latest', 'mistral:7b', 'minimax-m2.5:cloud']
Redis running. Memory used: 964.56K

Ollama: ✅   Redis: ✅


---
## Section 1 — Pydantic Cache Entry Validation

Every value written to or read from the cache is validated against `CacheEntry`.
This prevents silent data corruption — a bad write raises `ValidationError` immediately
rather than poisoning downstream LLM calls with garbage context.

```python
# Without validation: cache returns whatever was stored
cache['key'] = {'response': None, 'tokens': 'oops'}   # silent corruption

# With Pydantic: invalid writes raise ValidationError at the boundary
CacheEntry(response=None, tokens='oops')   # ValidationError: response cannot be None
```

In [10]:
class CacheTier(str, Enum):
    L1     = 'l1'
    L2     = 'l2'
    MISS   = 'miss'


class CacheEntry(BaseModel):
    key:          str
    response:     str             = Field(min_length=1)
    prompt_hash:  str             = Field(min_length=8)
    model:        str
    latency_s:    float           = Field(ge=0.0)
    tokens_est:   int             = Field(ge=0)
    created_at:   str
    ttl_seconds:  int             = Field(default=3600, ge=1)
    hit_count:    int             = Field(default=0, ge=0)
    tier:         CacheTier       = CacheTier.L1

    @field_validator('response')
    @classmethod
    def response_not_whitespace(cls, v: str) -> str:
        if not v.strip():
            raise ValueError('response cannot be empty or whitespace')
        return v.strip()

    @field_validator('prompt_hash')
    @classmethod
    def hash_looks_valid(cls, v: str) -> str:
        if not re.match(r'^[a-f0-9]+$', v):
            raise ValueError(f'prompt_hash must be hex, got: {v!r}')
        return v

    def is_expired(self) -> bool:
        created = datetime.fromisoformat(self.created_at)
        return datetime.now(timezone.utc) > created + timedelta(seconds=self.ttl_seconds)

    def to_redis(self) -> str:
        return self.model_dump_json()

    @classmethod
    def from_redis(cls, raw: str) -> 'CacheEntry':
        return cls.model_validate_json(raw)


def make_cache_key(prompt: str) -> tuple[str, str]:
    """Exact hash key — used as the storage key in L1/L2."""
    h = hashlib.sha256(prompt.strip().lower().encode()).hexdigest()
    return h[:16], h


# ═══════════════════════════════════════════════════════════════════════
# SemanticIndex — sits in front of L1/L2
# Embeds every stored prompt. On lookup, finds the most similar stored
# prompt via cosine similarity. If sim >= threshold, returns its cache
# key so L1/L2 can serve the hit — even for paraphrases.
#
# Flow:
#   'What is the capital of France?'  -> embed -> exact key stored
#   'Tell me the capital of france'   -> embed -> sim=0.94 >= 0.80 -> HIT
#   'What is the GDP of France?'      -> embed -> sim=0.51 <  0.80 -> MISS
# ═══════════════════════════════════════════════════════════════════════
from sentence_transformers import SentenceTransformer

print('Loading embedding model (all-mpnet-base-v2)...')
_embedder = SentenceTransformer('all-mpnet-base-v2')
print('Embedding model ready.')


class SemanticIndex:
    """
    In-memory vector index that maps prompt embeddings -> cache keys.
    Thread-safe. Shares the same short_key namespace as L1/L2.
    """
    def __init__(self, threshold: float = 0.80):
        self.threshold   = threshold
        self._lock       = threading.Lock()
        self._keys:  list[str]       = []   # short cache keys
        self._embs:  list[np.ndarray] = []  # corresponding embeddings
        self._prompts: list[str]     = []   # original prompts (for debug)
        self.semantic_hits   = 0
        self.semantic_misses = 0

    def _embed(self, text: str) -> np.ndarray:
        return _embedder.encode(text.strip(), normalize_embeddings=True)

    def add(self, prompt: str, short_key: str):
        """Index a new prompt -> key mapping."""
        emb = self._embed(prompt)
        with self._lock:
            # Avoid duplicate keys
            if short_key not in self._keys:
                self._keys.append(short_key)
                self._embs.append(emb)
                self._prompts.append(prompt)

    def find(self, prompt: str) -> tuple[Optional[str], float, str]:
        """
        Returns (short_key, similarity, matched_prompt).
        short_key is None on miss.
        """
        if not self._keys:
            self.semantic_misses += 1
            return None, 0.0, ''
        query_emb = self._embed(prompt)
        with self._lock:
            sims      = [float(np.dot(query_emb, e)) for e in self._embs]
            best_idx  = int(np.argmax(sims))
            best_sim  = sims[best_idx]
        if best_sim >= self.threshold:
            self.semantic_hits += 1
            return self._keys[best_idx], best_sim, self._prompts[best_idx]
        self.semantic_misses += 1
        return None, best_sim, ''

    def remove(self, short_key: str):
        with self._lock:
            if short_key in self._keys:
                idx = self._keys.index(short_key)
                self._keys.pop(idx)
                self._embs.pop(idx)
                self._prompts.pop(idx)

    def stats(self) -> dict:
        total = self.semantic_hits + self.semantic_misses
        return {
            'indexed': len(self._keys),
            'semantic_hits':   self.semantic_hits,
            'semantic_misses': self.semantic_misses,
            'semantic_hit_rate': f'{self.semantic_hits/total:.1%}' if total else 'N/A',
            'threshold': self.threshold,
        }


# ── Quick demo of SemanticIndex alone ────────────────────────────────────
print('\n=== SemanticIndex Demo ===\n')
idx = SemanticIndex(threshold=0.80)

canonical = 'What is the capital of France?'
short_key, _ = make_cache_key(canonical)
idx.add(canonical, short_key)

test_queries = [
    ('What is the capital of France?',      'exact match'),
    ('Can you give me the capital of France?', 'paraphrase'),
    ('Tell me the capital of france',        'lowercase paraphrase'),
    ('What is the capital city of France?', 'near-synonym'),
    ('What is the GDP of France?',           'different intent'),
    ('Who is the president of Germany?',     'different topic'),
]

for query, label in test_queries:
    key, sim, matched = idx.find(query)
    hit = key is not None
    icon = '✅ HIT ' if hit else '❌ MISS'
    print(f'  {icon}  sim={sim:.3f}  [{label}]')
    print(f'         query  : {query}')
    if hit:
        print(f'         matched: {matched}')
    print()

print(f'SemanticIndex stats: {idx.stats()}')


# ── Demo: validation catching bad data ───────────────────────────────────────
print('=== Pydantic Validation Demo ===\n')

good = CacheEntry(
    key='abc123', response='Paris is the capital of France.',
    prompt_hash='a1b2c3d4e5f6',  model=OLLAMA_MODEL,
    latency_s=0.42, tokens_est=12,
    created_at=datetime.now(timezone.utc).isoformat()
)
print(f'Valid entry created:  key={good.key}  response={good.response!r}')

bad_cases = [
    {'key': 'x', 'response': '',        'prompt_hash': 'a1b2c3d4', 'model': OLLAMA_MODEL,
     'latency_s': 0.1, 'tokens_est': 5, 'created_at': datetime.now(timezone.utc).isoformat()},
    {'key': 'x', 'response': 'ok',      'prompt_hash': 'NOT_HEX!', 'model': OLLAMA_MODEL,
     'latency_s': 0.1, 'tokens_est': 5, 'created_at': datetime.now(timezone.utc).isoformat()},
    {'key': 'x', 'response': 'ok',      'prompt_hash': 'a1b2c3d4', 'model': OLLAMA_MODEL,
     'latency_s': -1,  'tokens_est': 5, 'created_at': datetime.now(timezone.utc).isoformat()},
]
labels = ['empty response', 'invalid hash', 'negative latency']

for label, bad in zip(labels, bad_cases):
    try:
        CacheEntry(**bad)
        print(f'  {label}: no error (unexpected)')
    except ValidationError as e:
        msg = e.errors()[0]['msg']
        print(f'  {label}: ValidationError caught -> {msg}  ✅')


Loading embedding model (all-mpnet-base-v2)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9444.60it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model ready.

=== SemanticIndex Demo ===

  ✅ HIT   sim=1.000  [exact match]
         query  : What is the capital of France?
         matched: What is the capital of France?

  ✅ HIT   sim=0.883  [paraphrase]
         query  : Can you give me the capital of France?
         matched: What is the capital of France?

  ✅ HIT   sim=0.876  [lowercase paraphrase]
         query  : Tell me the capital of france
         matched: What is the capital of France?

  ✅ HIT   sim=0.977  [near-synonym]
         query  : What is the capital city of France?
         matched: What is the capital of France?

  ❌ MISS  sim=0.673  [different intent]
         query  : What is the GDP of France?

  ❌ MISS  sim=0.394  [different topic]
         query  : Who is the president of Germany?

SemanticIndex stats: {'indexed': 1, 'semantic_hits': 4, 'semantic_misses': 2, 'semantic_hit_rate': '66.7%', 'threshold': 0.8}
=== Pydantic Validation Demo ===

Valid entry created:  key=abc123  response='Paris is t

---
## Section 2 — L1 In-Memory Cache (LRU)

L1 is an `OrderedDict`-backed LRU cache capped at `max_size` entries.
Reads are sub-millisecond. When full, the least-recently-used entry is evicted.

```
get(key)  -> move to end (most recent) -> return entry
set(key)  -> if full: evict LRU (front) -> append to end
```

In [11]:
class L1Cache:
    def __init__(self, max_size: int = 128):
        self.max_size  = max_size
        self._store: OrderedDict[str, CacheEntry] = OrderedDict()
        self._lock     = threading.Lock()
        self.hits      = 0
        self.misses    = 0
        self.evictions = 0

    @property
    def size(self) -> int:
        return len(self._store)

    @property
    def utilisation(self) -> float:
        return self.size / self.max_size

    def get(self, key: str) -> Optional[CacheEntry]:
        with self._lock:
            if key not in self._store:
                self.misses += 1
                return None
            entry = self._store[key]
            if entry.is_expired():
                del self._store[key]
                self.misses += 1
                return None
            # Move to end = most recently used
            self._store.move_to_end(key)
            entry.hit_count += 1
            self.hits += 1
            return entry

    def set(self, key: str, entry: CacheEntry):
        with self._lock:
            if key in self._store:
                self._store.move_to_end(key)
            self._store[key] = entry
            if len(self._store) > self.max_size:
                evicted_key, _ = self._store.popitem(last=False)  # evict LRU
                self.evictions += 1

    def evict_lru(self, n: int = 1):
        with self._lock:
            for _ in range(min(n, len(self._store))):
                self._store.popitem(last=False)
                self.evictions += 1

    def clear_expired(self) -> int:
        with self._lock:
            expired = [k for k, v in self._store.items() if v.is_expired()]
            for k in expired:
                del self._store[k]
            return len(expired)

    def stats(self) -> dict:
        total = self.hits + self.misses
        return {
            'size': self.size, 'max_size': self.max_size,
            'utilisation': f'{self.utilisation:.1%}',
            'hits': self.hits, 'misses': self.misses,
            'hit_rate': f'{self.hits/total:.1%}' if total else 'N/A',
            'evictions': self.evictions,
        }


# Demo: fill cache past capacity and watch LRU eviction
print('=== L1 LRU Cache Demo ===\n')
l1 = L1Cache(max_size=5)

# Fill to capacity
for i in range(5):
    entry = CacheEntry(
        key=f'key_{i}', response=f'Response {i}', prompt_hash=f'abcdef{i:02d}1234',
        model=OLLAMA_MODEL, latency_s=0.1 * i, tokens_est=10 + i,
        created_at=datetime.now(timezone.utc).isoformat()
    )
    l1.set(f'key_{i}', entry)

print(f'After 5 inserts: size={l1.size}  (max=5)')

# Access key_0 to make it recently used
l1.get('key_0')
print(f'Accessed key_0 (now most-recently-used)')

# Insert key_5 -> should evict key_1 (LRU, since key_0 was just accessed)
l1.set('key_5', CacheEntry(
    key='key_5', response='Response 5', prompt_hash='abcdef051234',
    model=OLLAMA_MODEL, latency_s=0.5, tokens_est=15,
    created_at=datetime.now(timezone.utc).isoformat()
))

print(f'After 6th insert: size={l1.size}  evictions={l1.evictions}')
print(f'key_1 evicted?  {l1.get("key_1") is None}  ✅')
print(f'key_0 present?  {l1.get("key_0") is not None}  ✅  (was recently accessed)')
print(f'\nL1 stats: {l1.stats()}')


=== L1 LRU Cache Demo ===

After 5 inserts: size=5  (max=5)
Accessed key_0 (now most-recently-used)
After 6th insert: size=5  evictions=1
key_1 evicted?  True  ✅
key_0 present?  True  ✅  (was recently accessed)

L1 stats: {'size': 5, 'max_size': 5, 'utilisation': '100.0%', 'hits': 2, 'misses': 1, 'hit_rate': '66.7%', 'evictions': 1}


---
## Section 3 — L2 Redis Cache (with in-process fallback)

L2 persists entries across process restarts with TTL-based expiry.
If Redis is unavailable, falls back to a second in-memory `dict` so the notebook
stays runnable everywhere.

**Key design decisions:**
- Entries serialised as JSON via `CacheEntry.to_redis()` — Pydantic validates on read
- TTL set on the Redis key so eviction is automatic, no manual cleanup needed
- L2 miss promotes to L1 on read (write-through on the way back up)

In [12]:
class L2Cache:
    def __init__(self, default_ttl: int = 3600, prefix: str = 'agent_cache:'):
        self.default_ttl   = default_ttl
        self.prefix        = prefix
        self._fallback:    dict[str, CacheEntry] = {}   # used when Redis unavailable
        self.hits          = 0
        self.misses        = 0
        self.using_redis   = REDIS_AVAILABLE

    def _full_key(self, key: str) -> str:
        return f'{self.prefix}{key}'

    def get(self, key: str) -> Optional[CacheEntry]:
        fk = self._full_key(key)
        try:
            if self.using_redis and redis_client:
                raw = redis_client.get(fk)
                if raw is None:
                    self.misses += 1
                    return None
                entry = CacheEntry.from_redis(raw)
            else:
                entry = self._fallback.get(key)
                if entry is None:
                    self.misses += 1
                    return None

            if entry.is_expired():
                self.delete(key)
                self.misses += 1
                return None

            self.hits += 1
            entry.hit_count += 1
            return entry

        except (ValidationError, Exception) as e:
            print(f'L2 read error for {key}: {e}')
            self.misses += 1
            return None

    def set(self, key: str, entry: CacheEntry, ttl: Optional[int] = None):
        fk  = self._full_key(key)
        ttl = ttl or self.default_ttl
        try:
            if self.using_redis and redis_client:
                redis_client.setex(fk, ttl, entry.to_redis())
            else:
                self._fallback[key] = entry
        except Exception as e:
            print(f'L2 write error for {key}: {e}')

    def delete(self, key: str):
        if self.using_redis and redis_client:
            redis_client.delete(self._full_key(key))
        else:
            self._fallback.pop(key, None)

    def memory_bytes(self) -> int:
        if self.using_redis and redis_client:
            return redis_client.info('memory').get('used_memory', 0)
        # Estimate for fallback: rough JSON size
        return sum(len(e.to_redis().encode()) for e in self._fallback.values())

    def stats(self) -> dict:
        total = self.hits + self.misses
        return {
            'backend': 'redis' if self.using_redis else 'in-process fallback',
            'hits': self.hits, 'misses': self.misses,
            'hit_rate': f'{self.hits/total:.1%}' if total else 'N/A',
            'memory_bytes': self.memory_bytes(),
        }


print('=== L2 Cache Demo ===\n')
l2 = L2Cache(default_ttl=300)

entry = CacheEntry(
    key='demo_key', response='The speed of light is 299,792,458 m/s.',
    prompt_hash='c0ffee001234abcd', model=OLLAMA_MODEL,
    latency_s=1.2, tokens_est=18,
    created_at=datetime.now(timezone.utc).isoformat()
)

l2.set('demo_key', entry)
retrieved = l2.get('demo_key')
print(f'Write -> Read: {retrieved.response if retrieved else "MISS"}')
print(f'Backend in use: {l2.stats()["backend"]}')

# Validate that corrupted data in Redis is caught by Pydantic on read
if l2.using_redis and redis_client:
    redis_client.setex('agent_cache:corrupt_key', 60, '{"bad": "data"}')
    result = l2.get('corrupt_key')
    print(f'Corrupt Redis value handled gracefully: result={result}  ✅')

print(f'\nL2 stats: {l2.stats()}')


=== L2 Cache Demo ===

Write -> Read: The speed of light is 299,792,458 m/s.
Backend in use: redis
L2 read error for corrupt_key: 7 validation errors for CacheEntry
key
  Field required [type=missing, input_value={'bad': 'data'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
response
  Field required [type=missing, input_value={'bad': 'data'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
prompt_hash
  Field required [type=missing, input_value={'bad': 'data'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
model
  Field required [type=missing, input_value={'bad': 'data'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
latency_s
  Field required [type=missing, input_value={'bad': 'data'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing
tokens_est
  Field 

---
## Section 4 — Memory Monitor & Eviction Policy

`MemoryMonitor` samples L1 utilisation on a background thread.
When utilisation crosses a threshold, it triggers an eviction routine — no manual intervention.

**Eviction strategies implemented:**
- `LRU` — evict least-recently-used entries (default)
- `LFU` — evict least-frequently-used entries  
- `TTL_FIRST` — evict expired entries first, then fall back to LRU

**Why this matters:** Without active eviction, L1 fills up and every new entry
silently displaces a potentially hot entry. Monitoring makes the displacement explicit and controllable.

In [13]:
class EvictionPolicy(str, Enum):
    LRU       = 'lru'
    LFU       = 'lfu'
    TTL_FIRST = 'ttl_first'


@dataclass
class EvictionEvent:
    ts:              str
    trigger:         str            # 'threshold' | 'manual' | 'expired'
    policy:          EvictionPolicy
    entries_evicted: int
    util_before:     float
    util_after:      float


class MemoryMonitor:
    def __init__(self, l1: 'L1Cache', l2: 'L2Cache',
                 threshold: float = 0.75,
                 policy:    EvictionPolicy = EvictionPolicy.LRU,
                 check_interval_s: float = 5.0):
        self.l1               = l1
        self.l2               = l2
        self.threshold        = threshold
        self.policy           = policy
        self.check_interval_s = check_interval_s
        self.eviction_log:    list[EvictionEvent] = []
        self.util_history:    list[tuple[str, float]] = []
        self._running         = False
        self._thread:         Optional[threading.Thread] = None

    def _evict(self, trigger: str) -> int:
        util_before = self.l1.utilisation
        n_to_evict  = max(1, int(self.l1.size * 0.20))   # evict 20% at a time

        if self.policy == EvictionPolicy.TTL_FIRST:
            expired = self.l1.clear_expired()
            if expired < n_to_evict:
                self.l1.evict_lru(n_to_evict - expired)
            evicted = n_to_evict

        elif self.policy == EvictionPolicy.LFU:
            with self.l1._lock:
                sorted_entries = sorted(
                    self.l1._store.items(),
                    key=lambda kv: kv[1].hit_count
                )
                for k, _ in sorted_entries[:n_to_evict]:
                    del self.l1._store[k]
                    self.l1.evictions += 1
            evicted = n_to_evict

        else:   # LRU default
            self.l1.evict_lru(n_to_evict)
            evicted = n_to_evict

        evt = EvictionEvent(
            ts=datetime.now(timezone.utc).isoformat(),
            trigger=trigger, policy=self.policy,
            entries_evicted=evicted,
            util_before=util_before,
            util_after=self.l1.utilisation
        )
        self.eviction_log.append(evt)
        print(f'  [EVICTION] trigger={trigger}  policy={self.policy.value}  '
              f'evicted={evicted}  util {util_before:.0%} -> {self.l1.utilisation:.0%}')
        return evicted

    def check(self):
        util = self.l1.utilisation
        self.util_history.append((datetime.now(timezone.utc).isoformat(), util))
        if util >= self.threshold:
            print(f'  [MONITOR] L1 utilisation {util:.0%} >= threshold {self.threshold:.0%}  -> evicting')
            self._evict(trigger='threshold')

    def start(self):
        self._running = True
        def _loop():
            while self._running:
                self.check()
                time.sleep(self.check_interval_s)
        self._thread = threading.Thread(target=_loop, daemon=True)
        self._thread.start()

    def stop(self):
        self._running = False

    def report(self):
        print(f'\nMemory Monitor Report')
        print(f'  Current L1 utilisation : {self.l1.utilisation:.1%}')
        print(f'  Threshold              : {self.threshold:.0%}')
        print(f'  Policy                 : {self.policy.value}')
        print(f'  Eviction events        : {len(self.eviction_log)}')
        if self.eviction_log:
            total_evicted = sum(e.entries_evicted for e in self.eviction_log)
            print(f'  Total entries evicted  : {total_evicted}')
        print(f'  L1 stats               : {self.l1.stats()}')
        print(f'  L2 stats               : {self.l2.stats()}')


# ── Demo: fill L1 past threshold and watch monitor trigger eviction ───────────
print('=== Memory Monitor + Eviction Demo ===\n')

demo_l1  = L1Cache(max_size=20)
demo_l2  = L2Cache()
monitor  = MemoryMonitor(demo_l1, demo_l2, threshold=0.75,
                          policy=EvictionPolicy.LFU)

# Insert 16 entries (80% of 20) - above the 75% threshold
for i in range(16):
    e = CacheEntry(
        key=f'k{i:02d}', response=f'Cached response number {i}',
        prompt_hash=f'abcdef{i:04d}1234'[:16], model=OLLAMA_MODEL,
        latency_s=0.5, tokens_est=10,
        created_at=datetime.now(timezone.utc).isoformat()
    )
    # Simulate some entries being frequently accessed
    e.hit_count = i % 4   # entries 0,4,8,12 have hit_count=0 (LFU candidates)
    demo_l1.set(f'k{i:02d}', e)

print(f'L1 utilisation before check: {demo_l1.utilisation:.0%}  (threshold=75%)')
monitor.check()   # manually trigger a check
print(f'L1 utilisation after  check: {demo_l1.utilisation:.0%}')

monitor.report()

# Show all three policies side-by-side
print('\n--- Eviction policy comparison ---')
for policy in EvictionPolicy:
    test_l1 = L1Cache(max_size=10)
    for i in range(10):
        e = CacheEntry(
            key=f'p{i}', response=f'Response {i}',
            prompt_hash=f'abcdef{i:06d}1234'[:16], model=OLLAMA_MODEL,
            latency_s=0.1, tokens_est=5,
            created_at=datetime.now(timezone.utc).isoformat()
        )
        e.hit_count = i * 3
        test_l1.set(f'p{i}', e)
    m = MemoryMonitor(test_l1, L2Cache(), threshold=0.75, policy=policy)
    m._evict(trigger='demo')
    print(f'  {policy.value:12s}: {test_l1.size} entries remaining')


=== Memory Monitor + Eviction Demo ===

L1 utilisation before check: 80%  (threshold=75%)
  [MONITOR] L1 utilisation 80% >= threshold 75%  -> evicting
  [EVICTION] trigger=threshold  policy=lfu  evicted=3  util 80% -> 65%
L1 utilisation after  check: 65%

Memory Monitor Report
  Current L1 utilisation : 65.0%
  Threshold              : 75%
  Policy                 : lfu
  Eviction events        : 1
  Total entries evicted  : 3
  L1 stats               : {'size': 13, 'max_size': 20, 'utilisation': '65.0%', 'hits': 0, 'misses': 0, 'hit_rate': 'N/A', 'evictions': 3}
  L2 stats               : {'backend': 'redis', 'hits': 0, 'misses': 0, 'hit_rate': 'N/A', 'memory_bytes': 972720}

--- Eviction policy comparison ---
  [EVICTION] trigger=demo  policy=lru  evicted=2  util 100% -> 80%
  lru         : 8 entries remaining
  [EVICTION] trigger=demo  policy=lfu  evicted=2  util 100% -> 80%
  lfu         : 8 entries remaining
  [EVICTION] trigger=demo  policy=ttl_first  evicted=2  util 100% -> 80%


---
## Section 5 — Full Tiered Cache with Semantic Lookup

The complete lookup chain now has **4 layers**:

```
Incoming prompt
  └─> SemanticIndex  cosine_sim >= 0.80?  -> resolve to canonical key
        └─> L1  (in-process LRU)          -> sub-ms hit
              └─> L2  (Redis/fallback)     -> ~5ms hit, promotes to L1
                    └─> Ollama LLM         -> full inference, writes all layers
```

**What changes from before:**  
- `SemanticIndex` embeds every stored prompt using `all-mpnet-base-v2`  
- On each lookup, the query is embedded and compared against all stored embeddings  
- If `cosine_similarity >= threshold (0.80)`, the matched canonical key is used for L1/L2 lookup  
- *'What is the capital of France?'*, *'Tell me the capital of france'*, and  
  *'Can you give me the capital of France?'* all resolve to the **same cache entry**  

**Threshold tuning:**  
- Raise `semantic_threshold` (e.g. 0.90) → fewer hits, higher precision  
- Lower it (e.g. 0.70) → more hits, risk of wrong response for edge cases

In [14]:
class TieredCache:
    """
    Lookup order:
      1. SemanticIndex  — finds semantically equivalent cached prompt (cosine sim)
      2. L1             — sub-ms in-process LRU dict
      3. L2             — Redis (or in-process fallback), promotes to L1 on hit
      4. Ollama LLM     — on full miss; result indexed + written to L1 + L2
    """
    def __init__(self, l1_max: int = 128, l2_ttl: int = 3600,
                 eviction_threshold: float = 0.75,
                 semantic_threshold: float = 0.80,
                 policy: EvictionPolicy = EvictionPolicy.LRU):
        self.l1      = L1Cache(max_size=l1_max)
        self.l2      = L2Cache(default_ttl=l2_ttl)
        self.sem_idx = SemanticIndex(threshold=semantic_threshold)
        self.monitor = MemoryMonitor(self.l1, self.l2,
                                      threshold=eviction_threshold, policy=policy,
                                      check_interval_s=10.0)
        self.monitor.start()
        self._llm_calls    = 0
        self._total_calls  = 0

    def get(self, prompt: str, system: str = '') -> tuple[str, CacheTier, float, float]:
        """
        Returns (response, tier, elapsed_ms, semantic_similarity).
        similarity=1.0 for exact hits, <1.0 for semantic hits, 0.0 for misses.
        """
        t0 = time.time()
        self._total_calls += 1

        # Step 1: Semantic lookup — find closest cached prompt
        sem_key, sim, matched_prompt = self.sem_idx.find(prompt)

        if sem_key:
            # Step 2: Try L1 with the semantically matched key
            entry = self.l1.get(sem_key)
            if entry:
                return entry.response, CacheTier.L1, (time.time()-t0)*1000, sim

            # Step 3: Try L2 with the semantically matched key
            entry = self.l2.get(sem_key)
            if entry:
                entry.tier = CacheTier.L2
                self.l1.set(sem_key, entry)   # promote to L1
                return entry.response, CacheTier.L2, (time.time()-t0)*1000, sim

        # Step 4: Full miss — call Ollama
        self._llm_calls += 1
        response, latency_s = call_ollama(prompt, system=system)
        short_key, full_hash = make_cache_key(prompt)
        tokens_est = len(prompt.split()) + len(response.split())

        new_entry = CacheEntry(
            key=short_key, response=response,
            prompt_hash=full_hash, model=OLLAMA_MODEL,
            latency_s=latency_s, tokens_est=tokens_est,
            created_at=datetime.now(timezone.utc).isoformat()
        )
        # Write to L1, L2, and semantic index
        self.l1.set(short_key, new_entry)
        self.l2.set(short_key, new_entry)
        self.sem_idx.add(prompt, short_key)   # index the canonical form

        return response, CacheTier.MISS, (time.time()-t0)*1000, 0.0

    def stop(self):
        self.monitor.stop()

    def report(self):
        cached = self._total_calls - self._llm_calls
        print(f'\n{"="*58}')
        print('TIERED CACHE REPORT')
        print(f'{"="*58}')
        print(f'Total requests     : {self._total_calls}')
        print(f'LLM calls made     : {self._llm_calls}')
        print(f'Cache served       : {cached}  ({cached/self._total_calls:.0%})')
        print()
        print(f'Semantic index     : {self.sem_idx.stats()}')
        self.monitor.report()



# ── Run the benchmark ─────────────────────────────────────────────────────────
# Phase 1 — Cold start: canonical prompts go to Ollama, get indexed in SemanticIndex
# Phase 2 — Semantic hits: paraphrases resolve via embedding similarity, hit L1
# Phase 3 — L1 cleared: paraphrases now go L1-miss -> semantic hit -> L2 -> promote
# Phase 4 — TTL expiry: show expired entries fall back to Ollama correctly

print('=== Full Tiered Cache Demo (requires Ollama) ===\n')

tier_labels = {CacheTier.L1: '⚡ L1  ', CacheTier.L2: '🗄  L2  ', CacheTier.MISS: '🌐 LLM '}

cache = TieredCache(l1_max=32, l2_ttl=300,
                    eviction_threshold=0.75, semantic_threshold=0.80,
                    policy=EvictionPolicy.LRU)

def show(prompt, tier, ms, sim, note=''):
    tl  = tier_labels[tier]
    sim_str = f'sim={sim:.3f}' if sim > 0 else 'sim=---'
    print(f'{tl}  {ms:7.1f}ms  {sim_str}  {prompt[:48]}{"  <- " + note if note else ""}')

# ── Phase 1: Cold — canonical prompts, all miss ───────────────────────────
print(f'\n{"─"*60}')
print(' PHASE 1 — Cold start (all go to Ollama, get indexed)')
print(f'{"─"*60}')
CANONICAL = [
    'What is the capital of France?',
    'What is the capital of Japan?',
    'Explain what a hash function does in one sentence.',
]
for p in CANONICAL:
    r, tier, ms, sim = cache.get(p)
    show(p, tier, ms, sim)
    print(f'       {r[:90]}')
print(f'\nSemantic index now has {cache.sem_idx.stats()["indexed"]} entries.')

# ── Phase 2: Semantic hits — paraphrases of the same questions ───────────
print(f'\n{"─"*60}')
print(' PHASE 2 — Paraphrases (semantic match -> L1 hit, no Ollama call)')
print(f'{"─"*60}')
PARAPHRASES = [
    ('Can you give me the capital of France?',    'paraphrase of Q1'),
    ('Tell me the capital of france',             'lowercase paraphrase of Q1'),
    ('What is the capital city of France?',       'near-synonym of Q1'),
    ('What is the capital of Japan please?',      'polite paraphrase of Q2'),
    ('Describe japan capital',                    'terse paraphrase of Q2'),
    ('What does a hash function do?',             'paraphrase of Q3'),
]
for p, note in PARAPHRASES:
    r, tier, ms, sim = cache.get(p)
    show(p, tier, ms, sim, note)
    print(f'       {r[:90]}')

# ── Phase 3: L1 cleared — same paraphrases should hit L2 ─────────────────
print(f'\n{"─"*60}')
print(' PHASE 3 — L1 cleared (paraphrases -> semantic hit -> L2 -> promote to L1)')
print(f'{"─"*60}')
print('[SIM] Clearing L1 (simulates restart / eviction)...')
cache.l1._store.clear()
print(f'      L1 entries: {cache.l1.size}   SemanticIndex intact: {cache.sem_idx.stats()["indexed"]} entries')
for p, note in PARAPHRASES[:3]:
    r, tier, ms, sim = cache.get(p)
    show(p, tier, ms, sim, 'L2 -> L1 promote')

# ── Phase 4: Confirm different-intent queries still miss ──────────────────
print(f'\n{"─"*60}')
print(' PHASE 4 — Different intent (should miss semantic index -> go to Ollama)')
print(f'{"─"*60}')
DIFFERENT = [
    ('What is the GDP of France?',     'different intent'),
    ('Who is the president of Germany?','different topic'),
    ('What is 2 + 2?',                 'unrelated'),
]
for p, note in DIFFERENT:
    r, tier, ms, sim = cache.get(p)
    show(p, tier, ms, sim, note)

# ── Summary ───────────────────────────────────────────────────────────────
cache.report()
cache.stop()


=== Full Tiered Cache Demo (requires Ollama) ===


────────────────────────────────────────────────────────────
 PHASE 1 — Cold start (all go to Ollama, get indexed)
────────────────────────────────────────────────────────────
🌐 LLM    2762.5ms  sim=---  What is the capital of France?
       The capital of France is Paris.
🌐 LLM     414.4ms  sim=---  What is the capital of Japan?
       The capital of Japan is Tokyo.
🌐 LLM    1517.7ms  sim=---  Explain what a hash function does in one sentenc
       A hash function takes an input value of any size and generates a fixed-size output, known 

Semantic index now has 3 entries.

────────────────────────────────────────────────────────────
 PHASE 2 — Paraphrases (semantic match -> L1 hit, no Ollama call)
────────────────────────────────────────────────────────────
⚡ L1       13.9ms  sim=0.883  Can you give me the capital of France?  <- paraphrase of Q1
       The capital of France is Paris.
⚡ L1       11.8ms  sim=0.876  Tell me the capital o